In [1]:
# =====================================================
# 1. IMPORT LIBRARIES
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, LSTM, Dense, Embedding, GlobalMaxPooling1D, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping


# =====================================================
# 2. LOAD CLEANED DATASET
# =====================================================

df = pd.read_excel("/Users/mac/Desktop/enve/final enve/fill/final_filled_climate_cleaned.xlsx")

df["Date"] = pd.to_datetime(
    df["Date"],
    dayfirst=True,
    errors="coerce"
)

df = df.dropna(subset=["Date"])
df = df.sort_values(["Station", "Date"]).reset_index(drop=True)

print(df.head())
print(df.isnull().sum())


# =====================================================
# 3. FINAL QUALITY CHECK
# =====================================================

print("Pressure zeros:", (df["PRESS"] == 0).sum())
print("Negative rain:", (df["Rain"] < 0).sum())
print("RH > 100:", (df["RH"] > 100).sum())
print("RH < 0:", (df["RH"] < 0).sum())
print("Negative sunshine:", (df["SUNSHINE"] < 0).sum())

print(df.describe())


# =====================================================
# 4. TEMPORAL FEATURES
# =====================================================

df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Year"] = df["Date"].dt.year


# =====================================================
# 5. STATION ENCODING
# =====================================================

le = LabelEncoder()
df["Station_enc"] = le.fit_transform(df["Station"])

print("Station Encoding:")
for station, code in zip(le.classes_, range(len(le.classes_))):
    print(station, ":", code)


# =====================================================
# 6. DEFINE FEATURES AND TARGETS
# =====================================================

features = [
    "Temp",
    "RH",
    "Wind",
    "Rain",
    "PRESS",
    "SUNSHINE",
    "Month",
    "Week",
    "DayOfWeek"
]

targets = [
    "Temp",
    "RH",
    "Rain",
    "PRESS",
    "SUNSHINE"
]


# =====================================================
# 7. CHRONOLOGICAL SPLIT BEFORE SCALING
# =====================================================

train_df = df[df["Year"] <= 2022].copy()
val_df   = df[df["Year"] == 2023].copy()
test_df  = df[df["Year"] == 2024].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# =====================================================
# 8. SCALE DATA
# =====================================================

X_scaler = MinMaxScaler()
Y_scaler = MinMaxScaler()

X_scaler.fit(train_df[features])
Y_scaler.fit(train_df[targets])

train_X = X_scaler.transform(train_df[features])
val_X   = X_scaler.transform(val_df[features])
test_X  = X_scaler.transform(test_df[features])

train_Y = Y_scaler.transform(train_df[targets])
val_Y   = Y_scaler.transform(val_df[targets])
test_Y  = Y_scaler.transform(test_df[targets])

train_station = train_df["Station_enc"].values
val_station   = val_df["Station_enc"].values
test_station  = test_df["Station_enc"].values


# =====================================================
# 9. CREATE 30-DAY SEQUENCES
# =====================================================

SEQ_LEN = 30

def create_sequences(X, Y, station_ids, dates, seq_len=30):
    X_seq = []
    S_seq = []
    Y_seq = []
    D_seq = []

    for station in np.unique(station_ids):
        idx = np.where(station_ids == station)[0]

        X_station = X[idx]
        Y_station = Y[idx]
        dates_station = dates.iloc[idx].reset_index(drop=True)

        for i in range(len(X_station) - seq_len):
            X_seq.append(X_station[i:i + seq_len])
            S_seq.append(station)
            Y_seq.append(Y_station[i + seq_len])
            D_seq.append(dates_station.iloc[i + seq_len])

    return np.array(X_seq), np.array(S_seq), np.array(Y_seq), np.array(D_seq)


X_train, S_train, y_train, d_train = create_sequences(
    train_X, train_Y, train_station, train_df["Date"], SEQ_LEN
)

X_val, S_val, y_val, d_val = create_sequences(
    val_X, val_Y, val_station, val_df["Date"], SEQ_LEN
)

X_test, S_test, y_test, d_test = create_sequences(
    test_X, test_Y, test_station, test_df["Date"], SEQ_LEN
)

print("X_train:", X_train.shape)
print("S_train:", S_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("X_test:", X_test.shape)



    Station       Date  Temp     RH  Wind  Rain  PRESS  SUNSHINE  Month
0  HEB00008 2015-01-01   9.7   64.5   3.4   0.5  899.1       1.6      1
1  HEB00008 2015-01-02   6.6   92.0   3.0   0.0  901.3       5.0      1
2  HEB00008 2015-01-03   6.7   99.4   5.3  20.4  899.2       2.6      1
3  HEB00008 2015-01-04   5.1  100.0   4.0   2.0  897.7       1.0      1
4  HEB00008 2015-01-05   6.9   84.9   4.9   0.0  895.8       8.0      1
Station     0
Date        0
Temp        0
RH          0
Wind        0
Rain        0
PRESS       0
SUNSHINE    0
Month       0
dtype: int64
Pressure zeros: 0
Negative rain: 0
RH > 100: 0
RH < 0: 0
Negative sunshine: 0
                             Date          Temp            RH          Wind  \
count                       16768  16768.000000  16768.000000  16768.000000   
mean   2019-12-04 19:43:28.969465     20.024738     64.134244      3.230850   
min           2015-01-01 00:00:00     -0.400000      8.000000      0.000000   
25%           2017-04-19 00:00:00  

In [2]:
from tensorflow.keras.layers import (
    Input, Conv1D, GRU, Dense,
    Embedding, GlobalMaxPooling1D,
    Concatenate, Dropout
)

from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

# =====================================================
# PARAMETERS
# =====================================================

num_features = X_train.shape[2]
num_targets = y_train.shape[1]
num_stations = df["Station_enc"].nunique()

# =====================================================
# CNN-GRU MODEL
# =====================================================

seq_input = Input(
    shape=(30, num_features),
    name="sequence_input"
)

x = Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu",
    padding="same"
)(seq_input)

x = Conv1D(
    filters=32,
    kernel_size=3,
    activation="relu",
    padding="same"
)(x)

x = GRU(
    64,
    return_sequences=False
)(x)

x = Dropout(0.2)(x)

# =====================================================
# STATION EMBEDDING
# =====================================================

station_input = Input(
    shape=(1,),
    name="station_input"
)

s = Embedding(
    input_dim=num_stations,
    output_dim=4
)(station_input)

s = GlobalMaxPooling1D()(s)

# =====================================================
# MERGE
# =====================================================

combined = Concatenate()([x, s])

z = Dense(
    64,
    activation="relu"
)(combined)

z = Dropout(0.2)(z)

output = Dense(
    num_targets,
    activation="linear"
)(z)

cnn_gru_model = Model(
    inputs=[seq_input, station_input],
    outputs=output
)

cnn_gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

cnn_gru_model.summary()
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history = cnn_gru_model.fit(
    [X_train, S_train],
    y_train,
    validation_data=([X_val, S_val], y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)
y_pred_scaled_cnn_gru = cnn_gru_model.predict(
    [X_test, S_test]
)

y_pred_cnn_gru = Y_scaler.inverse_transform(
    y_pred_scaled_cnn_gru
)

y_true = Y_scaler.inverse_transform(y_test)

rain_idx = targets.index("Rain")

max_log_rain = max(
    np.nanmax(y_true[:, rain_idx]),
    np.nanmax(y_pred_cnn_gru[:, rain_idx])
)

y_true[:, rain_idx] = np.clip(
    y_true[:, rain_idx],
    0,
    max_log_rain
)

y_pred_cnn_gru[:, rain_idx] = np.clip(
    y_pred_cnn_gru[:, rain_idx],
    0,
    max_log_rain
)

y_true[:, rain_idx] = np.expm1(
    y_true[:, rain_idx]
)

y_pred_cnn_gru[:, rain_idx] = np.expm1(
    y_pred_cnn_gru[:, rain_idx]
)
# =====================================================
# CNN-GRU EVALUATION
# =====================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

cnn_gru_results = []

for i, target in enumerate(targets):

    mae = mean_absolute_error(
        y_true[:, i],
        y_pred_cnn_gru[:, i]
    )

    mse = mean_squared_error(
        y_true[:, i],
        y_pred_cnn_gru[:, i]
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_true[:, i],
        y_pred_cnn_gru[:, i]
    )

    cnn_gru_results.append({
        "Model": "CNN-GRU",
        "Target": target,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

cnn_gru_results_df = pd.DataFrame(
    cnn_gru_results
)

print(cnn_gru_results_df)

cnn_gru_results_df.to_excel(
    "CNN_GRU_Results.xlsx",
    index=False
)

print("Results saved as CNN_GRU_Results.xlsx")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 30, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 30, 64)    │      1,792 │ sequence_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 30, 32)    │      6,176 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ station_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 64)        │     18,816 │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 4)      │         20 │ station_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 4)         │          0 │ embedding[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 68)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,416 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 5)         │        325 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 31,545 (123.22 KB)

 Trainable params: 31,545 (123.22 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0241 - mae: 0.1118 - val_loss: 0.0125 - val_mae: 0.0752
Epoch 2/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0132 - mae: 0.0795 - val_loss: 0.0105 - val_mae: 0.0651
Epoch 3/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0118 - mae: 0.0730 - val_loss: 0.0093 - val_mae: 0.0552
Epoch 4/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0110 - mae: 0.0688 - val_loss: 0.0104 - val_mae: 0.0646
Epoch 5/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0106 - mae: 0.0667 - val_loss: 0.0093 - val_mae: 0.0563
Epoch 6/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0103 - mae: 0.0654 - val_loss: 0.0089 - val_mae: 0.0532
Epoch 7/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0099 - mae: 0.0636 - val_loss: 0.0094 - val_mae: 0.0581
Epoch 8/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0098 - mae: 0.0625 - val_loss: 0.0095 - val_mae: 0.0585
Epoch 9/100
388/388 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/